# 45 — Constantine neutron escape and recoverable D

One completed Roman traversal makes one desired neutron in
`C13 + alpha -> O16 + n`. Closure requires that enough of those
neutrons ultimately make recoverable D in ordinary hydrogen.

The key correction here is time dependence. A frozen compressed sphere
is millions of scattering paths thick, but it begins to disassemble
long before a room-temperature neutron would be captured. The reduced
model therefore follows neutron survival through homologous expansion
and adiabatic cooling, then exposes the still-unresolved target/H-blanket
albedo as a parameter.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
while repo_root.name != 'CNO' and repo_root != repo_root.parent:
    repo_root = repo_root.parent
source_root = repo_root / 'analysis' / 'src'
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
result_root = repo_root / 'analysis' / 'results' / 'roman-workbooks'
result_root.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 6)

from math import exp, sqrt
import cno_sweep.neutron_transport as nt
from cno_sweep import (
    Material, absorption_macroscopic_m1, constantine_efficiency_for_d_parity,
    driver_support_from_targets, evaluate_roman_closure, expanding_core_release,
    load_builtin_neutron_cross_sections, number_densities,
    repeated_blanket_capture_probability, roman_phase_one_closure_target_sets,
    roman_phase_one_target_sets,
)

xs = load_builtin_neutron_cross_sections()
workbook40_targets, _ = roman_phase_one_target_sets()
closure_targets = roman_phase_one_closure_target_sets()
supports = {case: driver_support_from_targets(case, targets) for case, targets in closure_targets.items()}

## Exact material threshold

With no credit from DT or DD neutrons, the required Constantine
neutron-to-recovered-D probability is exactly `5 P`, where `P` is the
total DT burn per completed traversal. Crediting auxiliary neutrons
lowers that threshold but never creates a second Constantine neutron.

In [2]:
threshold_rows = []
for case_name, support in supports.items():
    for auxiliary_eta in (0.0, 0.5, 0.8, 1.0):
        threshold_rows.append({
            'case': case_name,
            'eta_DT = eta_DD': auxiliary_eta,
            'DT burned P': support.total_dt_burned_per_traversal,
            'required eta_Constantine': constantine_efficiency_for_d_parity(
                support,
                eta_dt_n_to_d=auxiliary_eta,
                eta_dd_n_to_d=auxiliary_eta,
            ),
        })
thresholds = pd.DataFrame(threshold_rows)
display(thresholds.pivot(index='eta_DT = eta_DD', columns='case', values='required eta_Constantine'))

case,conservative,likely
eta_DT = eta_DD,,
0.0,0.685787,0.577042
0.5,0.548630,0.461633
0.8,0.466335,0.392388
1.0,0.411472,0.346225


## Why the static answer is wrong

The table compares the room-temperature absorption clock with the fuel
sound-crossing/disassembly clock. It is only a diagnostic: the actual
neutron distribution is initially MeV-hot and follows the cooling
plasma, so the expansion calculation below uses broad Maxwellian-group
averages rather than one 0.0253-eV cross section.

In [3]:
neutron_mass = 1.67492749804e-27
ev_j = 1.602176634e-19
thermal_speed = sqrt(2 * nt.THERMAL_CUTOFF_EV * ev_j / neutron_mass)
clock_rows = []
for case_name, targets in workbook40_targets.items():
    state = targets['constantine'].radius_state
    core = Material('C13/alpha', number_densities(
        state.compressed_density_kg_m3, {'c13': 1.0, 'he4': 1.0}, xs
    ))
    capture_time = 1.0 / (thermal_speed * absorption_macroscopic_m1(core, xs, nt.THERMAL_CUTOFF_EV))
    clock_rows.append({
        'case': case_name,
        'compressed R (m)': state.compressed_fuel_radius_m,
        'hydrodynamic time (s)': state.hydrodynamic_time_s,
        '0.0253-eV absorption time (s)': capture_time,
        'absorption/hydro time': capture_time / state.hydrodynamic_time_s,
        'static diffusion escape ~3L/R': min(1.0, 3 * nt.diffusion_length_m(core, xs) / state.compressed_fuel_radius_m),
    })
display(pd.DataFrame(clock_rows).set_index('case'))

,compressed R (m),hydrodynamic time (s),0.0253-eV absorption time (s),absorption/hydro time,static diffusion escape ~3L/R
case,,,,,
likely,1.343531,3.453365e-07,0.000063,183.242944,0.002820
conservative,16.122189,2.072019e-06,0.000095,45.810736,0.000352


## Correcting the Workbook-40 Constantine tie-break

Workbook 40 first minimized the largest target, then chose the
lowest-N15 card below that already-fixed ceiling. For Constantine the
N15 differences were tiny, so this selected a needlessly slow, large
160-keV target. The closure card keeps the same compression and burnup
but selects the highest temperature already present in each declared
grid. This is a local tie-break correction, not a global reoptimization.

In [4]:
card_rows = []
for case_name in closure_targets:
    for label, collection in [('Workbook 40', workbook40_targets), ('closure tie-break', closure_targets)]:
        target = collection[case_name]['constantine']
        state = target.radius_state
        card_rows.append({
            'case': case_name, 'card': label,
            'T_i (keV)': state.ion_temperature_keV,
            'compression': state.compression_ratio,
            'initial fuel R (m)': state.initial_fuel_equivalent_radius_m,
            'compressed fuel R (m)': state.compressed_fuel_radius_m,
            'complete target R (m)': target.physical_target_outer_radius_m,
            'hydro time (s)': state.hydrodynamic_time_s,
            'N15 burned/success': target.n15_burned_per_successful_reaction,
            'central DT burned/success': target.central_dt_burned_per_successful_reaction,
        })
display(pd.DataFrame(card_rows).set_index(['case', 'card']))

T_i (keV)  compression  initial fuel R (m)  \
case         card                                                            
likely       Workbook 40            160.0       3000.0           19.377065   
             closure tie-break      430.0       3000.0            0.332244   
conservative Workbook 40            160.0       2000.0          203.126847   
             closure tie-break      300.0       2000.0            9.125990   

                                compressed fuel R (m)  complete target R (m)  \
case         card                                                              
likely       Workbook 40                     1.343531              19.563751   
             closure tie-break               0.023036               0.336967   
conservative Workbook 40                    16.122189             207.528795   
             closure tie-break               0.724330               9.323820   

                                hydro time (s)  N15 burned/success  \
case         card                                                    
likely       Workbook 40          3.453365e-07            0.013274   
             closure tie-break    3.612020e-09            0.013327   
conservative Workbook 40          2.072019e-06            0.013137   
             closure tie-break    6.798501e-08            0.013137   

                                central DT burned/success  
case         card                                          
likely       Workbook 40                     1.327743e-07  
             closure tie-break               2.633952e-02  
conservative Workbook 40                     1.319038e-09  
             closure tie-break               1.454519e-05

In [5]:
release_rows = []
release_lookup = {}
for card_label, collection in [('Workbook 40', workbook40_targets), ('closure tie-break', closure_targets)]:
    for case_name, targets in collection.items():
        state = targets['constantine'].radius_state
        core = Material('C13/alpha', number_densities(
            state.compressed_density_kg_m3, {'c13': 1.0, 'he4': 1.0}, xs
        ))
        for multiplier in (0.5, 1.0, 3.0, 10.0, 30.0):
            result = expanding_core_release(
                core, xs, source_energy_mev=2.2,
                ion_temperature_keV=state.ion_temperature_keV,
                radius_m=state.compressed_fuel_radius_m,
                hydrodynamic_time_s=state.hydrodynamic_time_s,
                expansion_time_multiplier=multiplier,
            )
            release_lookup[(card_label, case_name, multiplier)] = result.release_probability
            release_rows.append({
                'card': card_label, 'case': case_name,
                'expansion time / hydro time': multiplier,
                'core release probability': result.release_probability,
                'decoupling scale R/Rf': result.decoupling_scale_factor,
                'neutron kT at decoupling (eV)': result.decoupling_energy_eV,
            })
release = pd.DataFrame(release_rows)
display(release.pivot_table(
    index=['card', 'expansion time / hydro time'], columns='case',
    values='core release probability'
))

case                                           conservative    likely
card              expansion time / hydro time                        
Workbook 40       0.5                              0.820231  0.947244
                  1.0                              0.672790  0.897286
                  3.0                              0.304549  0.722471
                  10.0                             0.019016  0.338564
                  30.0                             0.000020  0.039282
closure tie-break 0.5                              0.992565  0.999166
                  1.0                              0.985193  0.998341
                  3.0                              0.956257  0.995077
                  10.0                             0.861594  0.984077
                  30.0                             0.640263  0.955845

The broad Maxwellian group average matters: following one neutron
energy through individual resonances creates artificial spikes. The
expansion multiplier is the main unresolved hydrodynamic input. A
value of 1 means the radius doubles in one present sound-crossing time;
3–10 deliberately slows that expansion.

In [6]:
# A thermal neutron entering a thick H2 blanket one transport mean free path
# deep has this one-entry absorption probability in planar diffusion. Most of
# the remainder returns to the target; repeated low-loss returns can still
# approach complete capture.
h2 = Material('ordinary H2', number_densities(70.8, {'h1': 1.0}, xs), True)
h_mfp = nt.transport_mean_free_path_m(h2, xs)
h_diffusion = nt.diffusion_length_m(h2, xs)
capture_one_entry = 1.0 - exp(-h_mfp / h_diffusion)
albedo_rows = []
for target_return_survival in (0.0, 0.5, 0.8, 0.9, 0.95, 0.98, 0.99, 0.999):
    albedo_rows.append({
        'survival per return through target': target_return_survival,
        'eventual H capture': repeated_blanket_capture_probability(
            capture_one_entry, target_return_survival
        ),
    })
print('H transport mean free path (m):', h_mfp)
print('H diffusion length (m):', h_diffusion)
print('single-entry capture probability:', capture_one_entry)
display(pd.DataFrame(albedo_rows).set_index('survival per return through target'))

H transport mean free path (m): 0.0028476451833627714
H diffusion length (m): 0.025872561230092635
single-entry capture probability: 0.10422345952877832


,eventual H capture
survival per return through target,
0.000,0.104223
0.500,0.188772
0.800,0.367789
0.900,0.537786
0.950,0.699429
0.980,0.853318
0.990,0.920855
0.999,0.991478


## Reduced closure cases

The next table combines the expansion result with repeated H-blanket
encounters. DT-neutron efficiency is a declared transport input here;
Workbook 32 shows why Pb-208 is nearly transparent at 2.2 MeV but not
at 14.1 MeV. DD support neutrons can be generated directly in an H2
chamber and are therefore assigned separately.

In [7]:
scenarios = {
    'working likely': {
        'case': 'likely', 'expansion_multiplier': 3.0,
        'target_return_survival': 0.98, 'eta_dt': 0.75, 'eta_dd': 0.95,
    },
    'working conservative': {
        'case': 'conservative', 'expansion_multiplier': 10.0,
        'target_return_survival': 0.95, 'eta_dt': 0.55, 'eta_dd': 0.90,
    },
    'stress / not closed': {
        'case': 'conservative', 'expansion_multiplier': 30.0,
        'target_return_survival': 0.90, 'eta_dt': 0.35, 'eta_dd': 0.80,
    },
}
closure_rows = []
for scenario_name, scenario in scenarios.items():
    case_name = scenario['case']
    core_release = release_lookup[
        ('closure tie-break', case_name, scenario['expansion_multiplier'])
    ]
    blanket_capture = repeated_blanket_capture_probability(
        capture_one_entry, scenario['target_return_survival']
    )
    eta_constantine = core_release * blanket_capture
    ledger = evaluate_roman_closure(
        supports[case_name], eta_constantine_n_to_d=eta_constantine,
        eta_dt_n_to_d=scenario['eta_dt'], eta_dd_n_to_d=scenario['eta_dd'],
    )
    closure_rows.append({
        'scenario': scenario_name, **scenario,
        'core release': core_release,
        'eventual H capture after core': blanket_capture,
        'eta Constantine': eta_constantine,
        'D consumed': ledger.d_total_consumed,
        'D recovered': ledger.d_total_recovered,
        'net D': ledger.delta_d,
        'G_D': ledger.g_d,
    })
closure_table = pd.DataFrame(closure_rows).set_index('scenario')
display(closure_table)

,case,expansion_multiplier,target_return_survival,eta_dt,eta_dd,core release,eventual H capture after core,eta Constantine,D consumed,D recovered,net D,G_D
scenario,,,,,,,,,,,,
working likely,likely,3.0,0.98,0.75,0.95,0.995077,0.853318,0.849117,0.577042,1.045311,0.468270,1.811501
working conservative,conservative,10.0,0.95,0.55,0.90,0.861594,0.699429,0.602623,0.685787,0.801502,0.115714,1.168732
stress / not closed,conservative,30.0,0.90,0.35,0.80,0.640263,0.537786,0.344324,0.685787,0.502055,-0.183732,0.732086


## Checkpoint judgment

The reduced model now contains genuinely D-positive points in both the
likely and conservative target brackets. That is an **existence result
inside the workbook model**, not a transport-certified plant.

The result depends on three physical claims that the spatial campaign
must replace: (1) the closure-aware Constantine target expands on the
stated timescale, (2) repeated crossings of the burned driver/Pb-208
stack preserve the stated fraction, and (3) DD support is performed in
a geometry that really delivers its neutron to H. The stress row shows
that closure can still be lost; it is not algebraically guaranteed.

In [8]:
artifact = {
    'schema': 'roman-neutron-recovery-v0.1',
    'source_energy_mev': 2.2,
    'clock_rows': clock_rows,
    'card_rows': card_rows,
    'release_rows': release_rows,
    'h2': {
        'density_kg_m3': 70.8,
        'transport_mfp_m': h_mfp,
        'diffusion_length_m': h_diffusion,
        'capture_one_entry': capture_one_entry,
    },
    'closure_rows': closure_rows,
}
path = result_root / '45-neutron-and-d-recovery-v0.1.json'
path.write_text(json.dumps(artifact, indent=2))
print('wrote', path.relative_to(repo_root))

wrote analysis/results/roman-workbooks/45-neutron-and-d-recovery-v0.1.json
